In [113]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
            SparkSession
                .builder
                .master("local[*]")
                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark

In [145]:
dataframe = spark.read.parquet("/home/doug/project/spark/jobs/part-00000-40be3905-1643-4b96-8abc-31502a864176-c000.snappy.parquet")

In [146]:
#transform column name
colorList = dataframe.select(collect_set("color").alias("colors")).first()["colors"]

regex_pattern = r"\b("+ "|".join(colorList) +r")\b"

dataframe = (dataframe
                .withColumn("productname", split(col("productname"), ",")[0])
                .withColumn("productname", regexp_replace(col("productname"), regex_pattern, ""))
                .withColumn("productname", regexp_replace(col("productname"), " -", ""))
            )

dataframe

DataFrame[ProductID: int, productname: string, productnumber: string, productsubcategoryid: int, subcategoryname: string, productcategoryid: int, categoryname: string, color: string, class: string, size: string, productline: string, productmodelid: int, modelname: string, style: string, listprice: decimal(19,4), standardcost: decimal(19,4), sellstartdate: timestamp, sellenddate: timestamp, safetystocklevel: int, reorderpoint: int, sizeunitmeasurecode: string, weightunitmeasurecode: string, weight: decimal(8,2), daystomanufacture: int, discontinueddate: timestamp, inserted_date: timestamp, modified_date: string, eltversion: string]

In [148]:
#transform productmodel
dataframe = (dataframe
                .withColumn("productnumber",substring_index(col("productnumber"), "-", 2))
)
dataframe

DataFrame[ProductID: int, productname: string, productnumber: string, productsubcategoryid: int, subcategoryname: string, productcategoryid: int, categoryname: string, color: string, class: string, size: string, productline: string, productmodelid: int, modelname: string, style: string, listprice: decimal(19,4), standardcost: decimal(19,4), sellstartdate: timestamp, sellenddate: timestamp, safetystocklevel: int, reorderpoint: int, sizeunitmeasurecode: string, weightunitmeasurecode: string, weight: decimal(8,2), daystomanufacture: int, discontinueddate: timestamp, inserted_date: timestamp, modified_date: string, eltversion: string]

In [ ]:
#fillNullValues


dataframe = (dataframe
                .fillna({
                    
            })
            
        )